# Question 1: Sequential Task Pipeline
Task: Implement a 4-step document workflow using Gemini-powered ConversableAgents. The steps are:
DraftWriter → Writes a short article draft on a given topic.
EditorAgent → Refines the grammar, structure, and tone of the draft.
PlagiarismChecker → Checks if the content is original or copied.
PublisherAgent → Finalizes the article and prepares it for publishing.
Each agent should take the previous agent's output and continue the task without losing context.  Print each step’s output clearly. 
Use generate_reply() for all interactions.


In [2]:
# 1. Answer

from autogen import ConversableAgent

# Config for Gemini
config = {
    "config_list": [{
        "model": "models/gemini-3.5-flash-lite",
        "api_key": "AQ.Ab8RN6LpcX9M1P-V7JctBajwLVo-OUYBES2BPXcbSUNer1Ba5g",
        "api_type": "google"
    }]
}

# Define agents
draft_writer = ConversableAgent(name="DraftWriter", llm_config=config)
editor = ConversableAgent(name="EditorAgent", llm_config=config)
plagiarism_checker = ConversableAgent(name="PlagiarismChecker", llm_config=config)
publisher = ConversableAgent(name="PublisherAgent", llm_config=config)

# Step 1: DraftWriter creates a draft
draft = draft_writer.generate_reply(messages=[
    {"role": "user", "content": "Write a short article about the importance of clean drinking water."}
])

# Step 2: EditorAgent refines the draft
edited = editor.generate_reply(messages=[
    {"role": "user", "content": draft["content"]}
])

# Step 3: PlagiarismChecker checks for originality
plagiarism_check = plagiarism_checker.generate_reply(messages=[
    {"role": "user", "content": f"Check for plagiarism in the following text:\n\n{edited['content']}"}
])

# Step 4: PublisherAgent finalizes it for publishing
published = publisher.generate_reply(messages=[
    {"role": "user", "content": edited["content"]}
])

# Print results
print("Draft", draft["content"])
print("Edited", edited["content"])
print("Plagiarism Check", plagiarism_check["content"])
print("Published", published["content"])


>>>>>>>> USING AUTO REPLY...

>>>>>>>> USING AUTO REPLY...

>>>>>>>> USING AUTO REPLY...

>>>>>>>> USING AUTO REPLY...
Draft **The Elixir of Life: Why Clean Drinking Water is Irreplaceable**

Water is the foundation of all life. While we can survive for weeks without food, the human body can only last a few days without water. However, simply having access to liquid is not enough; that water must be clean. Clean drinking water is a fundamental human right, yet it remains one of the most critical global challenges of the 21st century.

First and foremost, clean water is essential for public health. The human body is roughly 60% water, and we rely on it to regulate temperature, flush out waste, lubricate joints, and transport nutrients. When water sources are contaminated with pathogens, bacteria, or heavy metals, they become vectors for deadly diseases such as cholera, dysentery, and typhoid. According to the World Health Organization (WHO), millions of people—predominantly children in

# Question 2. Task Decomposer (Planner + Workers)
Use a PlannerAgent to break down the goal: "Create a business plan for a startup in mental health tech". Then use MarketAgent, TechAgent, and FinancialAgent to address each part.


In [4]:
# 2. Answer

from autogen import ConversableAgent
import textwrap

# Config for Gemini
config = {
    "config_list": [{
        "model": "models/gemini-3.5-flash-lite",
        "api_key": "AQ.Ab8RN6LpcX9M1P-V7JctBajwLVo-OUYBES2BPXcbSUNer1Ba5g",
        "api_type": "google"
    }]
}

# Define agents
planner = ConversableAgent(name="PlannerAgent", llm_config=config)
market_agent = ConversableAgent(name="MarketAgent", llm_config=config)
tech_agent = ConversableAgent(name="TechAgent", llm_config=config)
finance_agent = ConversableAgent(name="FinancialAgent", llm_config=config)

# Utility function to trim long text
def truncate_text(text, max_words=200):
    words = text.split()
    return ' '.join(words[:max_words]) if len(words) > max_words else text

# Step 1: PlannerAgent breaks down the goal
planning = planner.generate_reply(messages=[
    {"role": "user", "content": "Break down the goal: Create a business plan for a startup in mental health tech"}
])

plan_text = planning["content"]
print("📋 Planner Breakdown", plan_text)

# Truncate if too long
short_plan = truncate_text(plan_text)

# Step 2: MarketAgent handles the market analysis
market = market_agent.generate_reply(messages=[
    {"role": "user", "content": f"From this plan, focus only on market analysis{short_plan}"}
])

# Step 3: TechAgent handles the technology plan
tech = tech_agent.generate_reply(messages=[
    {"role": "user", "content": f"From this plan, focus only on technology implementation{short_plan}"}
])

# Step 4: FinancialAgent handles the financial strategy
finance = finance_agent.generate_reply(messages=[
    {"role": "user", "content": f"From this plan, focus only on financial planning{short_plan}"}
])

# Print Outputs
print("📊 Market Analysis", market["content"])
print("💻 Tech Strategy", tech["content"])
print("💰 Financial Plan", finance["content"])


>>>>>>>> USING AUTO REPLY...
📋 Planner Breakdown Creating a business plan for a mental health tech startup requires a delicate balance between **commercial viability** and **clinical/ethical responsibility**. 

Here is a comprehensive, step-by-step breakdown of the goal, divided into logical phases, key actions, and deliverables.

---

### Phase 1: Discovery & Validation (The "Why" & "For Whom")
*Before writing a single word of the plan, you need to prove the idea works and is legally/ethically sound.*

*   **1.1 Define the Problem & Solution**
    *   Identify the specific pain point (e.g., therapist shortages, high cost of care, adolescent anxiety, workplace burnout).
    *   Define your tech solution (e.g., AI-driven CBT chatbot, B2B therapist-matching platform, VR exposure therapy).
*   **1.2 Conduct Market Research & Competitor Analysis**
    *   Map out competitors (e.g., BetterHelp, Talkspace, Woebot) and find your "moat" or Unique Value Proposition (UVP).
    *   Analyze marke

# Question 3. Role-Based Chat (Specialized Roles)
Question: Assign roles to agents:
DoctorAgent gives medical insight
DietitianAgent suggests a meal plan
FitnessCoachAgent suggests workouts For a given user query like: “Help me manage diabetes naturally.”

In [7]:
# 3. Answer

from autogen import ConversableAgent

# Gemini API config
config = {
    "config_list": [{
        "model": "models/gemini-3.5-flash-lite",
        "api_key": "AQ.Ab8RN6LpcX9M1P-V7JctBajwLVo-OUYBES2BPXcbSUNer1Ba5g",
        "api_type": "google"
    }]
}

# Define role-based agents
doctor = ConversableAgent(name="DoctorAgent", llm_config=config)
dietitian = ConversableAgent(name="DietitianAgent", llm_config=config)
fitness_coach = ConversableAgent(name="FitnessCoachAgent", llm_config=config)

# Shared user query
user_query = "Help me manage diabetes naturally."

# DoctorAgent gives medical insights
medical_reply = doctor.generate_reply(messages=[
    {"role": "user", "content": f"You are a medical professional. Provide insights for: {user_query}"}
])

# DietitianAgent suggests a meal plan
diet_reply = dietitian.generate_reply(messages=[
    {"role": "user", "content": f"You are a certified dietitian. Suggest a diabetes-friendly meal plan for: {user_query}"}
])

# FitnessCoachAgent suggests workouts
fitness_reply = fitness_coach.generate_reply(messages=[
    {"role": "user", "content": f"You are a fitness coach. Suggest suitable workouts for someone who asked: {user_query}"}
])

# Print each agent's specialized output
print("DoctorAgent (Medical Insight)", medical_reply["content"])
print("DietitianAgent (Meal Plan)", diet_reply["content"])
print("FitnessCoachAgent (Workout Suggestions)", fitness_reply["content"])


>>>>>>>> USING AUTO REPLY...

>>>>>>>> USING AUTO REPLY...

>>>>>>>> USING AUTO REPLY...
DoctorAgent (Medical Insight) Managing diabetes naturally involves a combination of lifestyle modifications aimed at stabilizing blood glucose levels, improving insulin sensitivity, and preventing long-term complications. 

*Disclaimer: As a medical professional, I must emphasize that natural management strategies are most effective when used as a complement to—not a replacement for—your prescribed medical treatment plan. Always consult your primary care physician or endocrinologist before making significant changes to your diet, exercise routine, or medication.*

Here are evidence-based insights on managing diabetes naturally:

### 1. Optimize Your Nutritional Intake
Diet is the single most impactful tool for managing blood sugar. 
*   **Focus on Complex Carbohydrates:** Simple carbohydrates (white bread, pastries, sugary drinks) cause rapid blood sugar spikes. Opt for high-fiber, complex carbs s

# Question 4. Multi-Language Translator Agents
Question: Input a paragraph in English. EnglishToSpanish, SpanishToFrench, and FrenchToGerman agents must translate it in chain. Output should be the final German translation.

In [9]:
# 4. Answer

from autogen import ConversableAgent

# Gemini API Configuration
config = {
    "config_list": [{
        "model": "models/gemini-3.5-flash-lite",
        "api_key": "AQ.Ab8RN6LpcX9M1P-V7JctBajwLVo-OUYBES2BPXcbSUNer1Ba5g",
        "api_type": "google"
    }]
}

# Create translator agents
eng_to_spanish = ConversableAgent(name="EnglishToSpanish", llm_config=config)
spanish_to_french = ConversableAgent(name="SpanishToFrench", llm_config=config)
french_to_german = ConversableAgent(name="FrenchToGerman", llm_config=config)

# Input paragraph
paragraph = "Nature is the best medicine. A walk in the forest or time spent near water can heal the soul and calm the mind."

# Step 1: English to Spanish
spanish_reply = eng_to_spanish.generate_reply(messages=[
    {"role": "user", "content": f"Translate this to Spanish: {paragraph}"}
])

# Step 2: Spanish to French
french_reply = spanish_to_french.generate_reply(messages=[
    {"role": "user", "content": f"Translate this to French: {spanish_reply['content']}"}
])

# Step 3: French to German
german_reply = french_to_german.generate_reply(messages=[
    {"role": "user", "content": f"Translate this to German: {french_reply['content']}"}
])

# Final Output
print("Final German Translation", german_reply["content"])



>>>>>>>> USING AUTO REPLY...

>>>>>>>> USING AUTO REPLY...

>>>>>>>> USING AUTO REPLY...
Final German Translation Here is the translation into German:

« Die Natur ist das beste Heilmittel. Ein Spaziergang im Wald oder ein Moment am Wasser kann die Seele heilen und den Geist beruhigen. »


# Question 5: Fact Checker Agent Flow (Truth vs. Fake Detection)
Task: Create a pipeline to verify a social media claim using 3 agents:
ClaimExtractorAgent: Extracts main claim from user input
FactCheckerAgent: Determines if it's real or fake
ReporterAgent: Outputs a news-style response

In [10]:
# 5. Answer

from autogen import ConversableAgent

config = {
    "config_list": [{
        "model": "models/gemini-3.5-flash-lite",
        "api_key": "AQ.Ab8RN6LpcX9M1P-V7JctBajwLVo-OUYBES2BPXcbSUNer1Ba5g",  
        "api_type": "google"
    }]
}

# Agents
extractor = ConversableAgent(name="ClaimExtractorAgent", llm_config=config)
checker = ConversableAgent(name="FactCheckerAgent", llm_config=config)
reporter = ConversableAgent(name="ReporterAgent", llm_config=config)

# User social media input
social_post = "I saw a post saying drinking hot water every 30 minutes kills COVID-19."

# Step 1: Extract the main claim
step1 = extractor.generate_reply(messages=[{"role": "user", "content": social_post}])

# Step 2: Check the factual accuracy
step2 = checker.generate_reply(messages=[{"role": "user", "content": step1["content"]}])

# Step 3: Generate a news-style report
step3 = reporter.generate_reply(messages=[{"role": "user", "content": step2["content"]}])

print("Extracted Claim", step1["content"])
print("Fact Check Result", step2["content"])
print("Final Report", step3["content"])


>>>>>>>> USING AUTO REPLY...

>>>>>>>> USING AUTO REPLY...

>>>>>>>> USING AUTO REPLY...
Extracted Claim That is **false**. Drinking hot water every 30 minutes does not kill the virus that causes COVID-19. 

This is a well-known health myth that has circulated widely on social media since the beginning of the pandemic. Here is what the science and health authorities say about it:

* **How the virus enters the body:** SARS-CoV-2 (the COVID-19 virus) enters your body primarily through the nose, mouth, and eyes, and then attaches to cells in your respiratory tract. 
* **The temperature problem:** Once the virus is inside your respiratory cells, drinking hot water will not reach it. Furthermore, to kill a virus inside the human body with heat, the water would have to be at a temperature that would severely burn and damage your throat and internal organs.
* **Lack of scientific evidence:** Major health organizations, including the World Health Organization (WHO) and the U.S. Centers for Di